[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/085.%20The%20Uncapacitated%20Facility%20Location%20Problem/P85-Tier-1_executed.ipynb)

---


# 85. The Uncapacitated Facility Location Problem
## Tier 1: A Constraint Satisfaction Formulation with Branch-and-Bound Search

### Purpose and scope

The uncapacitated facility location problem (UFLP) asks which facilities to open and how to assign customers to them. Opening a facility incurs a fixed cost; assigning a customer to a facility incurs a transportation cost. The goal is to minimize the sum of these costs while assigning every customer to exactly one open facility.

This notebook uses a constraint satisfaction problem (CSP) representation and a depth-first branch-and-bound search over facility-opening decisions. Constraint propagation rules out inconsistent partial decisions, and a lower bound allows the search to prune branches that cannot improve the best complete solution found so far. The method is exact for the stated model: when the search finishes, the retained solution has minimum cost.

The input is a fixed cost for each facility and a customer-by-facility transportation-cost matrix. The output is a facility-opening vector, one assigned facility per customer, a cost breakdown, and a verification of the constraints. The example has three facilities and four customers.

### Model assumptions

- Facilities have no capacity limits.
- Each customer must be assigned to exactly one facility.
- A customer can be assigned only to an open facility.
- Facility-opening and customer-assignment decisions are binary.
- Fixed costs and transportation costs are finite, nonnegative values in a common, unspecified cost unit.
- Each matrix entry is the total transportation cost for serving that customer from that facility. Customer demands are provided as contextual data but are not used to calculate costs; the example does not define a per-unit transportation rate or a demand-based cost formula.
- The transportation-cost matrix used below is an illustrative instance. No geographic coordinates, distances, or additional figure-derived values are assumed.

### Mathematical model

Let \(J=\{1,\ldots,m\}\) be the set of facilities and \(I=\{1,\ldots,n\}\) the set of customers. Let \(f_j\) be the fixed cost of opening facility \(j\), and let \(c_{ij}\) be the transportation cost of serving customer \(i\) from facility \(j\). Define binary variables \(y_j\) and \(x_{ij}\), where \(y_j=1\) means facility \(j\) is open and \(x_{ij}=1\) means customer \(i\) is assigned to facility \(j\).

The objective is to minimize total fixed and transportation cost:

$$
\min \quad \sum_{j\in J} f_j y_j + \sum_{i\in I}\sum_{j\in J} c_{ij}x_{ij}.
$$

Every customer is assigned exactly once, and assignments are linked to facility-opening decisions:

$$
\sum_{j\in J}x_{ij}=1 \quad (i\in I), \qquad
x_{ij}\leq y_j \quad (i\in I,\ j\in J), \qquad
x_{ij},y_j\in\{0,1\}.
$$

The first constraint prevents a customer from being left unserved or assigned to multiple facilities. The linking constraint prevents an assignment to a closed facility. There is no capacity constraint because facilities are uncapacitated. In the code, `fixed_costs` stores \(f_j\), `transport_costs[i, j]` stores \(c_{ij}\), `y_vars` represents facility decisions, and `x_vars` represents customer-facility assignment decisions.

### Search method

A partial search state records decisions that are fixed and decisions that remain unknown. Constraint propagation applies consequences that follow from those decisions. For example, closing a facility rules out every assignment to it; assigning a customer to a facility requires that facility to be open. If only one facility remains available to a customer, that assignment is forced.

Branch-and-bound explores the remaining facility decisions recursively. At each state, a lower bound estimates the least cost that any completion of the state could attain. The bound includes fixed costs for facilities already open and, for each customer, the cheapest transportation cost among facilities that are open or not yet decided. Because this estimate may allow a customer to use a facility that will later be closed, it can underestimate the cost of a completed solution, but cannot overestimate it. It is therefore safe for pruning.

If \(L\) is the lower bound at a state and \(U\) is the cost of the best complete solution found so far, a state can be pruned when \(L>U\). The implementation searches facility decisions in index order and retains one best solution; equal-cost optimal solutions need not all be returned.

For \(m\) facilities, there are at most \(2^m\) opening patterns. Evaluating a completed pattern and its best customer assignments takes \(O(nm)\) time, so the straightforward worst-case work is \(O(2^m nm)\), in addition to propagation overhead. Branch-and-bound can explore fewer patterns when its bounds are effective, but its worst-case time remains exponential in the number of facilities.

### Example data

The fixed costs are `[50, 80, 60]`. The accompanying demand values are `[15, 20, 25, 18]`; they are descriptive only under the cost interpretation stated above. The transportation costs are defined explicitly in the data cell below, with rows corresponding to customers and columns corresponding to facilities.

In [ ]:
from dataclasses import dataclass
from math import isclose, isfinite
from typing import List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


# The example is deterministic, so no random-number generator is required.
print("Libraries imported for the UFLP CSP formulation.")

In [ ]:
@dataclass
class UFLPInstance:
    """Input data for an uncapacitated facility location problem."""

    n_facilities: int
    n_customers: int
    fixed_costs: List[float]
    transport_costs: np.ndarray

    def __post_init__(self):
        """Check dimensions and cost assumptions before starting the search."""
        if self.n_facilities <= 0 or self.n_customers <= 0:
            raise ValueError("The instance must contain at least one facility and one customer.")
        if len(self.fixed_costs) != self.n_facilities:
            raise ValueError("The number of fixed costs must equal the number of facilities.")
        if self.transport_costs.shape != (self.n_customers, self.n_facilities):
            raise ValueError("Transportation costs must have shape (n_customers, n_facilities).")

        all_costs = np.concatenate(
            (np.asarray(self.fixed_costs, dtype=float), self.transport_costs.astype(float).ravel())
        )
        if not np.all(np.isfinite(all_costs)) or np.any(all_costs < 0):
            raise ValueError("All fixed and transportation costs must be finite and nonnegative.")


@dataclass
class UFLPSolution:
    """A feasible facility-opening and customer-assignment plan."""

    facilities_open: List[bool]
    assignments: List[int]
    total_cost: float
    fixed_cost: float
    transport_cost: float
    feasible: bool


print("UFLP data structures are ready.")

In [ ]:
def create_concrete_example():
    """Create the illustrative instance with three facilities and four customers."""
    n_facilities = 3
    n_customers = 4
    fixed_costs = [50, 80, 60]

    # Rows are customers and columns are facilities; entries are total service costs.
    transport_costs = np.array(
        [
            [12, 25, 18],
            [18, 8, 22],
            [25, 10, 15],
            [15, 9, 12],
        ],
        dtype=float,
    )

    return UFLPInstance(
        n_facilities=n_facilities,
        n_customers=n_customers,
        fixed_costs=fixed_costs,
        transport_costs=transport_costs,
    )


instance = create_concrete_example()

print("Created the illustrative UFLP instance:")
print(f"  Facilities: {instance.n_facilities}")
print(f"  Customers: {instance.n_customers}")
print(f"  Fixed costs: {instance.fixed_costs}")
print(f"  Transportation-cost matrix shape: {instance.transport_costs.shape}")

In [ ]:
class UFLPCSPSolver:
    """Solve a UFLP instance by CSP propagation and branch-and-bound search."""

    def __init__(self, instance: UFLPInstance):
        self.instance = instance
        self.best_solution = None
        self.nodes_explored = 0

    def constraint_propagation(self, y_vars, x_vars):
        """Propagate opening and assignment implications in a partial state.

        A value of None denotes an undecided binary variable. New lists are made
        so propagation in one search branch cannot alter a sibling branch.
        """
        y_vars = list(y_vars) if y_vars is not None else [None] * self.instance.n_facilities
        if x_vars is None:
            x_vars = [
                [None] * self.instance.n_facilities
                for _ in range(self.instance.n_customers)
            ]
        else:
            x_vars = [list(row) for row in x_vars]

        changed = True
        while changed:
            changed = False

            # A closed facility cannot receive any customer assignment.
            for j in range(self.instance.n_facilities):
                if y_vars[j] is False:
                    for i in range(self.instance.n_customers):
                        if x_vars[i][j] is True:
                            return None, None
                        if x_vars[i][j] is None:
                            x_vars[i][j] = False
                            changed = True

            for i in range(self.instance.n_customers):
                assigned_facilities = [
                    j for j in range(self.instance.n_facilities)
                    if x_vars[i][j] is True
                ]
                if len(assigned_facilities) > 1:
                    return None, None

                # An assignment requires its facility to be open, and excludes
                # every other assignment for the same customer.
                if assigned_facilities:
                    assigned_j = assigned_facilities[0]
                    if y_vars[assigned_j] is False:
                        return None, None
                    if y_vars[assigned_j] is None:
                        y_vars[assigned_j] = True
                        changed = True
                    for j in range(self.instance.n_facilities):
                        if j != assigned_j:
                            if x_vars[i][j] is True:
                                return None, None
                            if x_vars[i][j] is None:
                                x_vars[i][j] = False
                                changed = True
                    continue

                # If only one facility can still serve this customer, both the
                # assignment and the required facility opening are forced.
                available = [
                    j for j in range(self.instance.n_facilities)
                    if y_vars[j] is not False and x_vars[i][j] is not False
                ]
                if not available:
                    return None, None
                if len(available) == 1:
                    only_j = available[0]
                    x_vars[i][only_j] = True
                    if y_vars[only_j] is None:
                        y_vars[only_j] = True
                    changed = True

        return y_vars, x_vars

    def calculate_lower_bound(self, y_vars, x_vars):
        """Return a cost lower bound for all completions of a partial state."""
        lower_bound = sum(
            self.instance.fixed_costs[j]
            for j in range(self.instance.n_facilities)
            if y_vars[j] is True
        )

        for i in range(self.instance.n_customers):
            assigned = [
                j for j in range(self.instance.n_facilities)
                if x_vars[i][j] is True
            ]
            if assigned:
                lower_bound += self.instance.transport_costs[i, assigned[0]]
                continue

            possible_costs = [
                self.instance.transport_costs[i, j]
                for j in range(self.instance.n_facilities)
                if y_vars[j] is not False and x_vars[i][j] is not False
            ]
            if not possible_costs:
                return float("inf")
            lower_bound += min(possible_costs)

        return float(lower_bound)

    def branch_and_bound(self, y_vars, x_vars):
        """Explore facility-opening decisions, pruning infeasible or costly states."""
        self.nodes_explored += 1

        y_prop, x_prop = self.constraint_propagation(y_vars, x_vars)
        if y_prop is None:
            return None

        lower_bound = self.calculate_lower_bound(y_prop, x_prop)
        if self.best_solution is not None:
            best_cost = self.best_solution.total_cost
            if lower_bound > best_cost and not isclose(
                lower_bound, best_cost, rel_tol=1e-12, abs_tol=1e-12
            ):
                return None

        if all(value is not None for value in y_prop):
            if not any(y_prop):
                return None

            assignments = []
            transport_cost = 0.0
            for i in range(self.instance.n_customers):
                forced = [
                    j for j in range(self.instance.n_facilities)
                    if x_prop[i][j] is True
                ]
                if forced:
                    facility = forced[0]
                else:
                    open_facilities = [
                        j for j in range(self.instance.n_facilities)
                        if y_prop[j] is True
                    ]
                    facility = min(
                        open_facilities,
                        key=lambda j: self.instance.transport_costs[i, j],
                    )
                assignments.append(facility)
                transport_cost += self.instance.transport_costs[i, facility]

            fixed_cost = sum(
                self.instance.fixed_costs[j]
                for j in range(self.instance.n_facilities)
                if y_prop[j] is True
            )
            total_cost = fixed_cost + transport_cost
            solution = UFLPSolution(
                facilities_open=list(y_prop),
                assignments=assignments,
                total_cost=float(total_cost),
                fixed_cost=float(fixed_cost),
                transport_cost=float(transport_cost),
                feasible=True,
            )

            # Keep the first solution in a tie; the search is required to return
            # one optimum, not to enumerate every equivalent optimum.
            if self.best_solution is None or (
                solution.total_cost < self.best_solution.total_cost
                and not isclose(
                    solution.total_cost,
                    self.best_solution.total_cost,
                    rel_tol=1e-12,
                    abs_tol=1e-12,
                )
            ):
                self.best_solution = solution
            return solution

        # Branch on the first facility whose opening decision remains unknown.
        for j, decision in enumerate(y_prop):
            if decision is None:
                y_closed = list(y_prop)
                y_closed[j] = False
                self.branch_and_bound(y_closed, x_prop)

                y_open = list(y_prop)
                y_open[j] = True
                self.branch_and_bound(y_open, x_prop)
                break

        return self.best_solution

    def solve(self):
        """Run the search from a state in which all decisions are undecided."""
        self.best_solution = None
        self.nodes_explored = 0
        print("Starting CSP branch-and-bound search for the UFLP.")

        y_vars = [None] * self.instance.n_facilities
        x_vars = [
            [None] * self.instance.n_facilities
            for _ in range(self.instance.n_customers)
        ]
        solution = self.branch_and_bound(y_vars, x_vars)

        print(f"Search completed. Nodes explored: {self.nodes_explored}")
        return solution


print("The CSP branch-and-bound solver is defined.")

In [ ]:
solver = UFLPCSPSolver(instance)
solution = solver.solve()

if solution is None:
    raise RuntimeError("The search did not find a feasible solution.")

print("\n=== CSP SOLUTION ===")
print(f"Feasible: {solution.feasible}")
print(f"Total cost: {solution.total_cost:.1f} cost units")
print(f"Fixed cost: {solution.fixed_cost:.1f} cost units")
print(f"Transportation cost: {solution.transport_cost:.1f} cost units")
print(
    "\nFacilities opened: "
    f"{[j + 1 for j, is_open in enumerate(solution.facilities_open) if is_open]}"
)
print(
    "Customer assignments (facility number for each customer): "
    f"{[j + 1 for j in solution.assignments]}"
)

all_assigned = (
    len(solution.assignments) == instance.n_customers
    and all(0 <= facility < instance.n_facilities for facility in solution.assignments)
)
facility_consistency = all(
    solution.facilities_open[facility]
    for facility in solution.assignments
)
recomputed_fixed = sum(
    instance.fixed_costs[j]
    for j, is_open in enumerate(solution.facilities_open)
    if is_open
)
recomputed_transport = sum(
    instance.transport_costs[i, facility]
    for i, facility in enumerate(solution.assignments)
)

assert all_assigned
assert facility_consistency
assert isclose(recomputed_fixed, solution.fixed_cost, rel_tol=1e-12, abs_tol=1e-12)
assert isclose(
    recomputed_transport,
    solution.transport_cost,
    rel_tol=1e-12,
    abs_tol=1e-12,
)
assert isclose(
    solution.fixed_cost + solution.transport_cost,
    solution.total_cost,
    rel_tol=1e-12,
    abs_tol=1e-12,
)

print("\n=== CONSTRAINT AND COST CHECKS ===")
print(f"Every customer is assigned exactly once: {all_assigned}")
print(f"Every assigned facility is open: {facility_consistency}")
print("Fixed, transportation, and total costs are consistent: True")
print(f"Search nodes explored: {solver.nodes_explored}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle("UFLP CSP Solution Analysis", fontsize=16, fontweight="bold")

# The cost breakdown shows how fixed and customer-service costs contribute
# to the objective value of the selected plan.
ax1 = axes[0, 0]
ax1.pie(
    [solution.fixed_cost, solution.transport_cost],
    labels=["Fixed cost", "Transportation cost"],
    colors=["#2E86AB", "#A23B72"],
    autopct="%1.1f%%",
    startangle=90,
)
ax1.set_title("Total Cost Breakdown")

# Facility bars encode binary opening decisions; the legend explains the colors.
ax2 = axes[0, 1]
facility_numbers = [f"F{j + 1}" for j in range(instance.n_facilities)]
open_values = [int(is_open) for is_open in solution.facilities_open]
bar_colors = [
    "#2E86AB" if is_open else "#C73E1D"
    for is_open in solution.facilities_open
]
ax2.bar(facility_numbers, open_values, color=bar_colors)
ax2.set_title("Facility Opening Decisions")
ax2.set_ylabel("Decision (1 = open, 0 = closed)")
ax2.set_ylim(0, 1.15)
ax2.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color="#2E86AB", label="Open"),
        plt.Rectangle((0, 0), 1, 1, color="#C73E1D", label="Closed"),
    ],
    loc="upper right",
)
for j, value in enumerate(open_values):
    ax2.text(j, value + 0.04, f"Fixed cost: {instance.fixed_costs[j]:g}", ha="center")

# A value of one marks the unique facility assigned to each customer.
ax3 = axes[1, 0]
assignment_matrix = np.zeros((instance.n_customers, instance.n_facilities))
for i, facility in enumerate(solution.assignments):
    assignment_matrix[i, facility] = 1
sns.heatmap(
    assignment_matrix,
    annot=True,
    fmt=".0f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    cbar_kws={"label": "Assignment (1 = selected)"},
    xticklabels=[f"F{j + 1}" for j in range(instance.n_facilities)],
    yticklabels=[f"C{i + 1}" for i in range(instance.n_customers)],
    ax=ax3,
)
ax3.set_title("Customer-Facility Assignment Matrix")
ax3.set_xlabel("Facilities")
ax3.set_ylabel("Customers")

# The final panel displays the input cost matrix; outlined cells identify the
# transportation costs actually used by the returned assignment.
ax4 = axes[1, 1]
sns.heatmap(
    instance.transport_costs,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    cbar_kws={"label": "Transportation cost (cost units)"},
    xticklabels=[f"F{j + 1}" for j in range(instance.n_facilities)],
    yticklabels=[f"C{i + 1}" for i in range(instance.n_customers)],
    ax=ax4,
)
for i, facility in enumerate(solution.assignments):
    ax4.add_patch(plt.Rectangle((facility, i), 1, 1, fill=False, edgecolor="blue", linewidth=3))
ax4.set_title("Transportation-Cost Matrix and Selected Assignments")
ax4.set_xlabel("Facilities")
ax4.set_ylabel("Customers")

plt.tight_layout()
plt.show()

print("\n=== ASSIGNMENT COST DETAILS ===")
print("Customer -> Facility (transportation cost)")
for i, facility in enumerate(solution.assignments):
    cost = instance.transport_costs[i, facility]
    print(f"Customer {i + 1} -> Facility {facility + 1} (cost: {cost:g})")

In [ ]:
def sensitivity_analysis(instance):
    """Re-solve the same UFLP under controlled cost-multiplier experiments."""
    print("=== SENSITIVITY ANALYSIS ===")

    fixed_cost_multipliers = [0.8, 0.9, 1.0, 1.1, 1.2]
    fixed_cost_results = []
    print("\n1. Fixed-cost sensitivity:")

    for multiplier in fixed_cost_multipliers:
        modified_instance = UFLPInstance(
            n_facilities=instance.n_facilities,
            n_customers=instance.n_customers,
            fixed_costs=[cost * multiplier for cost in instance.fixed_costs],
            transport_costs=instance.transport_costs.copy(),
        )
        result = UFLPCSPSolver(modified_instance).solve()
        if result is None:
            raise RuntimeError("No feasible solution was found in a sensitivity experiment.")

        fixed_cost_results.append(
            {
                "multiplier": multiplier,
                "total_cost": result.total_cost,
                "facilities_open": sum(result.facilities_open),
                "fixed_cost": result.fixed_cost,
            }
        )
        print(
            f"  {multiplier:.1f}x: total cost = {result.total_cost:.1f}, "
            f"facilities open = {sum(result.facilities_open)}"
        )

    transport_multipliers = [0.8, 0.9, 1.0, 1.1, 1.2]
    transport_results = []
    print("\n2. Transportation-cost sensitivity:")

    for multiplier in transport_multipliers:
        modified_instance = UFLPInstance(
            n_facilities=instance.n_facilities,
            n_customers=instance.n_customers,
            fixed_costs=list(instance.fixed_costs),
            transport_costs=instance.transport_costs * multiplier,
        )
        result = UFLPCSPSolver(modified_instance).solve()
        if result is None:
            raise RuntimeError("No feasible solution was found in a sensitivity experiment.")

        transport_results.append(
            {
                "multiplier": multiplier,
                "total_cost": result.total_cost,
                "facilities_open": sum(result.facilities_open),
                "transport_cost": result.transport_cost,
            }
        )
        print(
            f"  {multiplier:.1f}x: total cost = {result.total_cost:.1f}, "
            f"facilities open = {sum(result.facilities_open)}"
        )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    fixed_multipliers = [row["multiplier"] for row in fixed_cost_results]
    fixed_totals = [row["total_cost"] for row in fixed_cost_results]
    fixed_open_counts = [row["facilities_open"] for row in fixed_cost_results]
    ax1.plot(fixed_multipliers, fixed_totals, "o-", color="#2E86AB", label="Total cost")
    ax1.set_xlabel("Fixed-cost multiplier")
    ax1.set_ylabel("Total cost (cost units)", color="#2E86AB")
    ax1.tick_params(axis="y", labelcolor="#2E86AB")
    ax1.set_title("Fixed-Cost Sensitivity")
    ax1.grid(True, alpha=0.3)
    ax1_twin = ax1.twinx()
    ax1_twin.plot(
        fixed_multipliers,
        fixed_open_counts,
        "s--",
        color="#A23B72",
        label="Facilities open",
    )
    ax1_twin.set_ylabel("Number of facilities open", color="#A23B72")
    ax1_twin.tick_params(axis="y", labelcolor="#A23B72")
    ax1.legend(loc="upper left")
    ax1_twin.legend(loc="upper right")

    transport_x = [row["multiplier"] for row in transport_results]
    transport_totals = [row["total_cost"] for row in transport_results]
    transport_open_counts = [row["facilities_open"] for row in transport_results]
    ax2.plot(transport_x, transport_totals, "o-", color="#2E86AB", label="Total cost")
    ax2.set_xlabel("Transportation-cost multiplier")
    ax2.set_ylabel("Total cost (cost units)", color="#2E86AB")
    ax2.tick_params(axis="y", labelcolor="#2E86AB")
    ax2.set_title("Transportation-Cost Sensitivity")
    ax2.grid(True, alpha=0.3)
    ax2_twin = ax2.twinx()
    ax2_twin.plot(
        transport_x,
        transport_open_counts,
        "s--",
        color="#A23B72",
        label="Facilities open",
    )
    ax2_twin.set_ylabel("Number of facilities open", color="#A23B72")
    ax2_twin.tick_params(axis="y", labelcolor="#A23B72")
    ax2.legend(loc="upper left")
    ax2_twin.legend(loc="upper right")

    plt.tight_layout()
    plt.show()

    return fixed_cost_results, transport_results


fixed_cost_results, transport_results = sensitivity_analysis(instance)

In [ ]:
print("=== INTERPRETING THE CSP SOLUTION ===")
print(
    "The returned plan minimizes the modeled fixed and transportation costs "
    "over the facility-opening decisions explored by branch-and-bound."
)
print(
    "For each open-facility pattern, the solution assigns each customer to "
    "the least-cost available open facility, subject to any assignment forced "
    "by constraint propagation."
)
print(
    "The sensitivity experiments vary one cost category at a time. Changes "
    "in total cost show the direct effect of the parameter change; changes "
    "in the number of open facilities indicate that the preferred network "
    "structure has also changed."
)
print(
    "The model does not include facility capacities, demand-dependent rates, "
    "facility-location coordinates, uncertainty, or other operational constraints."
)

## Strengths, limitations, and interpretation

### Strengths

- The branch-and-bound search provides an optimal solution for the stated finite instance when it terminates.
- Constraint propagation detects inconsistent partial decisions and forces decisions that follow logically from the model.
- The lower bound supports safe pruning without excluding a strictly better feasible solution.
- The cost breakdown and feasibility checks make the returned plan auditable.

### Limitations

- The number of facility-opening patterns can grow exponentially with the number of facilities, so this exact search is intended for small instances.
- The example treats each transportation-cost entry as a total customer-facility cost. Demand does not affect the objective unless the input costs are constructed to include it.
- The model omits capacities, geographic distance calculations, uncertainty, and other operational considerations; conclusions apply only to the assumptions and data represented here.
- When several plans have the same minimum cost, the search returns one of them rather than listing every optimum.

### How to read the results

The open-facility list describes the network selected by the search. The assignment list gives one facility index for each customer, using one-based numbering in the printed report. Fixed cost is the sum of opening costs for selected facilities; transportation cost is the sum of the assigned customer-facility costs. Their sum is the objective value. The verification checks confirm that every customer is assigned once, every assigned facility is open, and the reported cost components agree with the plan.